In [ ]:
from vectorbt.returns.nb import *

# get_return_nb
计算单期收益率 $\frac{{output\_value - input\_value}}{{input\_value}}$

```python
@njit(cache=True)
def get_return_nb(input_value: float, output_value: float) -> float:

    if input_value == 0:
        if output_value == 0:
            return 0.
        return np.inf * np.sign(output_value)
    return_value = (output_value - input_value) / input_value
    if input_value < 0:
        return_value *= -1
    return return_value
```

In [ ]:
print(get_return_nb(100.0, 110.0))  # 10%上涨
print(get_return_nb(100.0, 90.0))   # 10%下跌
print(get_return_nb(0.0, 10.0))     # 从零开始
print(get_return_nb(-100.0, -90.0)) # 负基数情况

# returns_

## returns_1d_nb
计算多期收益率
- 第 i 期收益率 = get_return_nb(第i-1期价格, 第i期价格)
- 第 0 期收益率 = get_return_nb(初始价格, 第0期价格)

```python
@njit(cache=True)
def returns_1d_nb(value: tp.Array1d, init_value: float) -> tp.Array1d:

    out = np.empty(value.shape, dtype=np.float64)
    input_value = init_value
    for i in range(out.shape[0]):
        output_value = value[i]
        out[i] = get_return_nb(input_value, output_value)
        input_value = output_value
    return out
```

In [ ]:
prices = np.array([100.0, 110.0, 105.0, 115.0])
returns_1d_nb(prices, 95.0)

## returns_nb
计算多资产多期收益率

```python
@njit(cache=True)
def returns_nb(value: tp.Array2d, init_value: tp.Array1d) -> tp.Array2d:

    out = np.empty(value.shape, dtype=np.float64)
    for col in range(out.shape[1]):
        out[:, col] = returns_1d_nb(value[:, col], init_value[col])
    return out
```

In [ ]:
prices = np.array([[100.0, 200.0],   # t0: 股票A=100, 股票B=200
                    [110.0, 190.0],   # t1: 股票A=110, 股票B=190
                    [105.0, 210.0]])  # t2: 股票A=105, 股票B=210
init_prices = np.array([95.0, 180.0])  # 初始基准价格
returns_nb(prices, init_prices)

# total_return_apply_nb
计算总收益率 $\mathop \Pi \limits_i \left( {1 + {r_i}} \right) - 1$

```python
@njit(cache=True)
def total_return_apply_nb(idxs: tp.Array1d, col: int, returns: tp.Array1d) -> float:
    return np.nanprod(returns + 1) - 1
```

In [ ]:
returns = np.array([0.1, -0.05, 0.08, 0.02])
print(total_return_apply_nb(None, None, returns))
# 计算过程：
# (1+0.1) × (1-0.05) × (1+0.08) × (1+0.02) - 1

# `cum_returns_`

## cum_returns_1d_nb
一维累计收益率计算：
- start_value ≠ 0：累计收益率[i] = (1+r[0]) × (1+r[1]) × ... × (1+r[i]) × start_value
- start_value = 0：累计收益率[i] = (1+r[0]) × (1+r[1]) × ... × (1+r[i]) - 1

```python
@njit(cache=True)
def cum_returns_1d_nb(returns: tp.Array1d, start_value: float) -> tp.Array1d:
    out = np.empty_like(returns, dtype=np.float64)
    cumprod = 1
    for i in range(returns.shape[0]):
        if not np.isnan(returns[i]):
            cumprod *= returns[i] + 1
        out[i] = cumprod
    if start_value == 0.:
        return out - 1.
    return out * start_value
```

In [ ]:
returns = np.array([0.1, -0.05, 0.08, 0.02])
print(cum_returns_1d_nb(returns, 0.0))  # 相对累计收益
print(cum_returns_1d_nb(returns, 100.0))  # 绝对资产价值

## cum_returns_nb
二维累计收益率计算

```python
@njit(cache=True)
def cum_returns_nb(returns: tp.Array2d, start_value: float) -> tp.Array2d:
    out = np.empty_like(returns, dtype=np.float64)
    for col in range(returns.shape[1]):
        out[:, col] = cum_returns_1d_nb(returns[:, col], start_value)
    return out
```

In [ ]:
returns = np.array([[0.1, 0.05],    # t1: 资产A=10%, 资产B=5%
                     [-0.05, 0.02],  # t2: 资产A=-5%, 资产B=2%
                     [0.08, -0.01]]) # t3: 资产A=8%, 资产B=-1%
cum_returns_nb(returns, 100.0)

## cum_returns_final_1d_nb
等价于 `cum_returns_1d_nb` 的最后一个值：
- start_value ≠ 0：(1+r[0]) × (1+r[1]) × ... × (1+r[end]) × start_value
- start_value = 0： (1+r[0]) × (1+r[1]) × ... × (1+r[end]) - 1

```python
@njit(cache=True)
def cum_returns_final_1d_nb(returns: tp.Array1d, start_value: float = 0.) -> float:

    out = np.nanprod(returns + 1.)
    if start_value == 0.:
        return out - 1.
    return out * start_value
```

## cum_returns_final_nb
等价于 `cum_returns_nb` 的最后一行值：

```python
@njit(cache=True)
def cum_returns_final_1d_nb(returns: tp.Array1d, start_value: float = 0.) -> float:

    out = np.nanprod(returns + 1.)
    if start_value == 0.:
        return out - 1.
    return out * start_value
```

In [ ]:
returns = np.array([[0.1, 0.05],
                     [-0.05, 0.02],
                     [0.08, -0.01]])
print(cum_returns_final_nb(returns, 0.0))

## rolling_cum_returns_final_nb
计算 `returns` 每列滑动窗口长度为 `window` 的累计收益率

参数
- `returns` (tp.Array2d): 二维收益率矩阵，形状为(时间点数, 资产数)
- `window` (int): 滚动窗口大小（观测期数）
- `minp` (tp.Optional[int]): 最小有效观测期数
  - 当窗口内有效数据少于此值时返回NaN
  - None 时使用 window 作为最小期数
- `start_value` (float): 起始价值，默认为 0（相对收益率）

返回：tp.Array2d，滚动累计收益率矩阵，形状与输入相同

```python
@njit
def rolling_cum_returns_final_nb(returns: tp.Array2d,
                                 window: int,
                                 minp: tp.Optional[int],
                                 start_value: float = 0.) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _start_value):
        return cum_returns_final_1d_nb(_returns, _start_value)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, start_value)
```

In [ ]:
returns = np.array([[0.02, 0.01],
                     [0.03, -0.01],
                     [-0.01, 0.02],
                     [0.01, 0.01]])
print(rolling_cum_returns_final_nb(returns, window=3, minp=2, start_value=0.0))

# `annualized_return_`

## annualized_return_1d_nb
根据日频/周频/月频收益率序列 `returns` 计算年化收益率

参数
- `returns` (tp.Array1d)：一维收益率时间序列，可以是日频/周频/月频
- `ann_factor` (float)：年化因子
  - 日频：通常为252（年均交易日）
  - 周频：通常为52（年均周数）
  - 月频：通常为12（年均月数）

原理：$年化收益率 = {\left( {1 + returns累计收益率} \right)^{\frac{{{\rm{ann}}\_factor}}{{len(returns)}}}} - 1$

返回：float，年化收益率（小数形式，如 0.15 表示 15%）

```python
@njit(cache=True)
def annualized_return_1d_nb(returns: tp.Array1d, ann_factor: float) -> float:
    end_value = cum_returns_final_1d_nb(returns, 1.)
    return end_value ** (ann_factor / returns.shape[0]) - 1
```

In [ ]:
returns = np.array([0.02, 0.01, -0.005, 0.03])  # 4个月收益率
print(annualized_return_1d_nb(returns, 12.0))  # 月频数据年化
# 计算过程：
# 1. 累计收益率 = (1.02) × (1.01) × (0.995) × (1.03) - 1
# 2. 年化收益率 = (1 + 累计收益率)^(12/4) - 1

## annualized_return_nb
二维版本的 `annualized_return_1d_nb`

```python
@njit(cache=True)
def annualized_return_nb(returns: tp.Array2d, ann_factor: float) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = annualized_return_1d_nb(returns[:, col], ann_factor)
    return out
```

In [ ]:
returns = np.array([[0.01, 0.02, -0.005],   # 资产A, B, C的月收益率
                     [0.02, -0.01, 0.01],
                     [-0.005, 0.015, 0.02],
                     [0.03, 0.01, -0.01]])
print(annualized_return_nb(returns, 12.0))  # 各资产的年化收益率

## rolling_annualized_return_nb
计算 `returns` 每列滑动窗口长度为 `window` 的年化收益率

参数
- `returns` (tp.Array2d): 二维收益率矩阵，形状为(时间点数, 资产数)
- `window` (int): 滚动窗口大小（观测期数）
- `minp` (tp.Optional[int]): 最小有效观测期数
  - 当窗口内有效数据少于此值时返回NaN
  - None 时使用 window 作为最小期数
- `ann_factor` (float)：年化因子
  - 日频：通常为252（年均交易日）
  - 周频：通常为52（年均周数）
  - 月频：通常为12（年均月数）

返回：tp.Array2d，滚动年化收益率矩阵，形状与输入相同

```python
@njit
def rolling_annualized_return_nb(returns: tp.Array2d,
                                 window: int,
                                 minp: tp.Optional[int],
                                 ann_factor: float) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _ann_factor):
        return annualized_return_1d_nb(_returns, _ann_factor)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, ann_factor)
```

# annualized_volatility_

## annualized_volatility_1d_nb
计算年化波动率
$$\sqrt {\frac{{\sum\limits_{1 \le i \le n} {{{\left( {{x_i} - \bar x} \right)}^2}} }}{{n - ddof}}}  \cdot ann\_facto{r^{\frac{1}{{levy\_alpha}}}}$$

参数
- `returns` (tp.Array1d): 一维收益率时间序列，可以是日频/周频/月频
- `ann_factor` (float): 年化因子
  - 日频数据：252（年均交易日）
  - 月频数据：12（年均月数）
  - 周频数据：52（年均周数）
- `levy_alpha` (float): Levy稳定性指数，默认为2.0（正态分布）
- `ddof` (int): 自由度调整，默认为1（样本标准差）

返回值：float，年化波动率（小数形式，如0.15表示15%）

```python
@njit(cache=True)
def annualized_volatility_1d_nb(returns: tp.Array1d,
                                ann_factor: float,
                                levy_alpha: float = 2.0,
                                ddof: int = 1) -> float:
    if returns.shape[0] < 2:
        return np.nan
    return generic_nb.nanstd_1d_nb(returns, ddof) * ann_factor ** (1.0 / levy_alpha)
```

In [ ]:
returns = np.array([0.02, -0.01, 0.015, -0.005, 0.01])
print(annualized_volatility_1d_nb(returns, 252.0, 2.0, 1))

## annualized_volatility_nb
二维版本的 `annualized_volatility_1d_nb`

```python
@njit(cache=True)
def annualized_volatility_nb(returns: tp.Array2d,
                             ann_factor: float,
                             levy_alpha: float = 2.0,
                             ddof: int = 1) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = annualized_volatility_1d_nb(returns[:, col], ann_factor, levy_alpha, ddof)
    return out
```

In [ ]:
returns = np.array([[0.01, 0.02, -0.005],   # 3个资产的日收益率
                     [0.02, -0.01, 0.01],
                     [-0.005, 0.015, 0.02],
                     [0.03, 0.01, -0.01]])
print(annualized_volatility_nb(returns, 252.0, 2.0, 1))

## rolling_annualized_volatility_nb
计算 `returns` 每列滑动窗口长度为 `window` 的年化收益率

参数
- `returns` (tp.Array2d): 二维收益率矩阵
- `window` (int): 滚动窗口大小
- `minp` (tp.Optional[int]): 最小有效观测期数
- `ann_factor` (float): 年化因子
- `levy_alpha` (float): Levy稳定性指数，默认2.0
- `ddof` (int): 自由度调整，默认1

返回值：tp.Array2d，滚动年化波动率矩阵，形状与输入相同

```python
@njit
def rolling_annualized_volatility_nb(returns: tp.Array2d,
                                     window: int,
                                     minp: tp.Optional[int],
                                     ann_factor: float,
                                     levy_alpha: float = 2.0,
                                     ddof: int = 1) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _ann_factor, _levy_alpha, _ddof):
        return annualized_volatility_1d_nb(_returns, _ann_factor, _levy_alpha, _ddof)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, ann_factor, levy_alpha, ddof)
```

In [ ]:
returns = np.random.normal(0, 0.02, (252, 2))  # 252个交易日，2个资产
rolling_vol = rolling_annualized_volatility_nb(
     returns, window=30, minp=20, ann_factor=252.0
)
print(rolling_vol)

# drawdown_1

## drawdown_1d_nb
计算：$回撤率[i] = \frac{{累计净值[i]}}{{\max \left\{ {累计净值[0], \cdots ,累计净值[i]} \right\}}} - 1$
- 其中 $累计净值\left[ i \right] = \left( {1 + r\left[ 0 \right]} \right) \times  \cdots  \times \left( {1 + r\left[ i \right]} \right)$

```python
@njit(cache=True)
def drawdown_1d_nb(returns: tp.Array1d) -> tp.Array1d:
    cum_returns = cum_returns_1d_nb(returns, start_value=100.)
    max_returns = generic_nb.expanding_max_1d_nb(cum_returns, minp=1)
    return cum_returns / max_returns - 1
```

In [ ]:
returns = np.array([0.1, -0.05, -0.02, 0.08])
print(drawdown_1d_nb(returns))

## drawdown_nb
二维版本的 `drawdown_1d_nb`

```python
@njit(cache=True)
def drawdown_nb(returns: tp.Array2d) -> tp.Array2d:
    out = np.empty_like(returns, dtype=np.float64)
    for col in range(returns.shape[1]):
        out[:, col] = drawdown_1d_nb(returns[:, col])
    return out
```

In [ ]:
returns = np.array([[0.02, 0.01],    # 两个策略的收益率
                     [-0.03, -0.01],
                     [0.01, 0.02]])
print(drawdown_nb(returns))

# max_drawdown_

## max_drawdown_1d_nb
计算 $最大回撤 = \mathop {\min }\limits_i \left\{ {回撤率[i]} \right\}$

```python
@njit(cache=True)
def max_drawdown_1d_nb(returns: tp.Array1d) -> float:
    return np.min(drawdown_1d_nb(returns))
```

In [ ]:
returns = np.array([0.1, -0.15, -0.05, 0.2, -0.08])
print(max_drawdown_1d_nb(returns))

## max_drawdown_nb
二维版本的 `max_drawdown_1d_nb`

```python
@njit(cache=True)
def max_drawdown_nb(returns: tp.Array2d) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = max_drawdown_1d_nb(returns[:, col])
    return out
```

In [ ]:
returns = np.array([[0.05, 0.02, 0.08],    # 3个策略
                     [-0.1, -0.02, -0.05],
                     [-0.05, 0.03, 0.02],
                     [0.15, -0.01, -0.03]])
print(max_drawdown_nb(returns))

## rolling_max_drawdown_nb
计算 `returns` 每列滑动窗口长度为 `window` 的最大回撤

参数
- `returns` (tp.Array2d): 二维收益率矩阵
- `window` (int): 滚动窗口大小
- `minp` (tp.Optional[int]): 最小有效观测期数

返回：tp.Array2d，滚动最大回撤矩阵，形状与输入相同

```python
@njit
def rolling_max_drawdown_nb(returns: tp.Array2d, window: int, minp: tp.Optional[int]) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns):
        return max_drawdown_1d_nb(_returns)
    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb)
```

In [ ]:
returns = np.random.normal(0.001, 0.02, (100, 2))  # 100期，2个策略
rolling_mdd = rolling_max_drawdown_nb(returns, window=20, minp=10)
print(rolling_mdd)

# calmar_ratio_
计算 $Calmar Ratio = \frac{年化收益率}{{\left| 最大回撤 \right|}}$。其中 
- $年化收益率 = {\left( {1 + returns累计收益率} \right)^{\frac{{{\rm{ann}}\_factor}}{{len(returns)}}}} - 1$
- $最大回撤 = \mathop {\min }\limits_i \left\{ {回撤率[i]} \right\}$

## calmar_ratio_1d_nb

```python
@njit(cache=True)
def calmar_ratio_1d_nb(returns: tp.Array1d, ann_factor: float) -> float:
    max_drawdown = max_drawdown_1d_nb(returns)
    if max_drawdown == 0.:
        return np.nan
    annualized_return = annualized_return_1d_nb(returns, ann_factor)
    if max_drawdown == 0.:
        return np.inf
    return annualized_return / np.abs(max_drawdown)
```

In [ ]:
returns = np.array([0.02, 0.01, -0.01, 0.03, -0.02])  # 月收益率
print(calmar_ratio_1d_nb(returns, 12.0))

## calmar_ratio_nb
二维版本的 `calmar_ratio_1d_nb`

```python
@njit(cache=True)
def calmar_ratio_nb(returns: tp.Array2d, ann_factor: float) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = calmar_ratio_1d_nb(returns[:, col], ann_factor)
    return out
```

In [ ]:
returns = np.array([[0.02, 0.01, 0.03],   # 3个策略的月收益率
                     [0.01, -0.02, 0.01],
                     [-0.01, 0.03, -0.02],
                     [0.03, -0.01, 0.04]])
print(calmar_ratio_nb(returns, 12.0))

## rolling_calmar_ratio_nb
计算 `returns` 每列滑动窗口长度为 `window` 的 Calmar Ratio

参数
- `returns` (tp.Array2d): 二维收益率矩阵
- `window` (int): 滚动窗口大小
- `minp` (tp.Optional[int]): 最小有效观测期数
- `ann_factor` (float): 年化因子

返回：tp.Array2d，滚动 Calmar Ratio 矩阵，形状与输入相同

```python
@njit
def rolling_calmar_ratio_nb(returns: tp.Array2d,
                            window: int,
                            minp: tp.Optional[int],
                            ann_factor: float) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _ann_factor):
        return calmar_ratio_1d_nb(_returns, _ann_factor)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, ann_factor)

```

In [ ]:
returns = np.random.normal(0.005, 0.02, (252, 3))  # 252个交易日，3个策略
rolling_calmar = rolling_calmar_ratio_nb(
     returns, window=60, minp=30, ann_factor=252.0
)
print(rolling_calmar)

# omega_ratio_
计算 Omega ratio
- 将目标年化收益率 `required_return` 转换为对应频率（日频/周频/月频/年频）的收益率
  - ${\left( {1 + x} \right)^{ann\_factor}} = 1 + required\_return$
- 计算 超额收益 = `returns` - `risk_free` - x
  - 其中 `risk_free` 为无风险收益率（频率应当和 `returns` 一致）
- 计算 *正向超额收益总和* 和 *负向超额收益总和*
- $OmegaRatio = \frac{正向超额收益总和}{负向超额收益总和}$

参数
- `returns` (tp.Array1d): 一维收益率时间序列
- `ann_factor` (float): 年化因子，用于将目标收益率转换为对应频率
  - 日频数据：252（年均交易日数）
  - 周频数据：52（年均周数）
  - 月频数据：12（年均月数）
  - 特殊值1：表示不进行年化转换，直接使用 required_return
- `risk_free` (float, 可选): 无风险收益率，默认为0
  - 应与returns的频率保持一致
- `required_return` (float, 可选): 目标年化收益率，默认为0

返回值：float
- 大于1：表示策略表现优于目标收益率，值越大越好
- 等于1：表示策略刚好达到目标收益率
- 小于1：表示策略表现低于目标收益率
- 无穷大：表示没有负超额收益，策略完美

## omega_ratio_1d_nb

```python
@njit(cache=True)
def omega_ratio_1d_nb(returns: tp.Array1d,
                      ann_factor: float,
                      risk_free: float = 0.,
                      required_return: float = 0.) -> float:
    if ann_factor == 1:
        return_threshold = required_return
    elif ann_factor <= -1:
        return np.nan
    else:
        return_threshold = (1 + required_return) ** (1. / ann_factor) - 1
    returns_less_thresh = returns - risk_free - return_threshold
    numer = np.sum(returns_less_thresh[returns_less_thresh > 0.0])
    denom = -1.0 * np.sum(returns_less_thresh[returns_less_thresh < 0.0])
    if denom == 0.:
        return np.inf
    return numer / denom
```

In [ ]:
daily_returns = np.array([0.001, -0.002, 0.003, 0.0, -0.001])
risk_free_daily = 0.03 / 252  # 年化3%转为日频
omega = omega_ratio_1d_nb(daily_returns, 252.0, risk_free_daily, 0.10)
print(f"考虑无风险利率的欧米伽比率: {omega:.3f}")

## omega_ratio_nb
二维版本的 `omega_ratio_1d_nb`

```python
@njit(cache=True)
def omega_ratio_nb(returns: tp.Array2d,
                   ann_factor: float,
                   risk_free: float = 0.,
                   required_return: float = 0.) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = omega_ratio_1d_nb(
            returns[:, col], ann_factor, risk_free, required_return)
    return out
```

In [ ]:
returns = np.array([[0.02, 0.01, 0.03],   # 3个策略的月收益率
                     [0.01, -0.02, 0.01],
                     [-0.01, 0.03, -0.02],
                     [0.03, -0.01, 0.04]])
print(omega_ratio_nb(returns, 12.0, 0.0, 0.08))

## rolling_omega_ratio_nb
计算 `returns` 每列滑动窗口长度为 `window` 的 Omega ratio

参数
- `returns` (tp.Array2d): 二维收益率矩阵，形状为(时间点数, 资产数)
- `window` (int): 滚动窗口大小（时间点数）
- `minp` (tp.Optional[int]): 窗口内所需的最小有效观测数，可选
- `ann_factor` (float): 年化因子，用于所有窗口计算
  - 日频数据：252（年均交易日数）
  - 周频数据：52（年均周数）
  - 月频数据：12（年均月数）
- `risk_free` (float, 可选): 无风险收益率，默认为0
- `required_return` (float, 可选): 目标年化收益率，默认为0

返回：tp.Array2d，滚动 Omega ratio 矩阵，形状与输入相同

```python
@njit
def rolling_omega_ratio_nb(returns: tp.Array2d,
                           window: int,
                           minp: tp.Optional[int],
                           ann_factor: float,
                           risk_free: float = 0.,
                           required_return: float = 0.) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _ann_factor, _risk_free, _required_return):
        return omega_ratio_1d_nb(_returns, _ann_factor, _risk_free, _required_return)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, ann_factor, risk_free, required_return)
```

In [ ]:
monthly_returns = np.array([
     [0.02, 0.015, 0.025],   # 3个基金的月度收益率
     [0.01, -0.005, 0.018],
     [-0.008, 0.022, -0.012],
     [0.025, 0.008, 0.030],
     [0.012, 0.015, 0.008],
     [0.018, -0.002, 0.022],
     [0.005, 0.020, 0.015],
     [-0.003, 0.012, -0.005],
     [0.028, 0.006, 0.035]
])
rolling_omega = rolling_omega_ratio_nb(
    monthly_returns, window=6, minp=4, ann_factor=12.0,
    risk_free=0.0, required_return=0.08
)
print(rolling_omega)

# sharpe_ratio_

## sharpe_ratio_1d_nb
计算 $年化夏普比 = \frac{平均超额收益}{超额收益标准差} \cdot \sqrt {年化因子} $
- 其中 超额收益 = `returns` - 无风险利率`risk_free`
- `ddof` 的作用参考 `annualized_volatility_`

```python
@njit(cache=True)
def sharpe_ratio_1d_nb(returns: tp.Array1d,
                       ann_factor: float,
                       risk_free: float = 0.,
                       ddof: int = 1) -> float:
    if returns.shape[0] < 2:
        return np.nan
    returns_risk_adj = returns - risk_free
    mean = np.nanmean(returns_risk_adj)
    std = generic_nb.nanstd_1d_nb(returns_risk_adj, ddof)
    if std == 0.:
        return np.inf
    return mean / std * np.sqrt(ann_factor)
```

In [ ]:
returns = np.array([0.02, 0.01, -0.01, 0.03, -0.005])  # 月收益率
print(sharpe_ratio_1d_nb(returns, 12.0, 0.002, 1))

## sharpe_ratio_nb
二维版本的 `sharpe_ratio_1d_nb`

```python
@njit(cache=True)
def sharpe_ratio_nb(returns: tp.Array2d,
                    ann_factor: float,
                    risk_free: float = 0.,
                    ddof: int = 1) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = sharpe_ratio_1d_nb(returns[:, col], ann_factor, risk_free, ddof)
    return out

```

In [ ]:
returns = np.array([[0.01, 0.02, 0.005],   # 3个资产的日收益率
                    [0.015, -0.01, 0.01],
                    [-0.005, 0.03, 0.008],
                    [0.02, -0.005, 0.012]])
print(sharpe_ratio_nb(returns, 252.0, 0.0001, 1))

## rolling_sharpe_ratio_nb
计算 `returns` 每列滑动窗口长度为 `window` 的 Omega ratio

参数
- `returns` (tp.Array2d): 二维收益率矩阵
- `window` (int): 滚动窗口大小（建议至少30个观测点）
- `minp` (tp.Optional[int]): 最小有效观测期数
- `ann_factor` (float): 年化因子
- `risk_free` (float): 无风险利率，默认为0
- `ddof` (int): 自由度调整，默认为1

返回值：tp.Array2d，滚动夏普比率矩阵，形状与输入相同

```python
@njit
def rolling_sharpe_ratio_nb(returns: tp.Array2d,
                            window: int,
                            minp: tp.Optional[int],
                            ann_factor: float,
                            risk_free: float = 0.,
                            ddof: int = 1) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _ann_factor, _risk_free, _ddof):
        return sharpe_ratio_1d_nb(_returns, _ann_factor, _risk_free, _ddof)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, ann_factor, risk_free, ddof)
```

In [ ]:
returns = np.random.normal(0.0005, 0.015, (252, 2))  # 252交易日，2个策略
rolling_sharpe = rolling_sharpe_ratio_nb(
     returns, window=60, minp=30, ann_factor=252.0, risk_free=0.0001
)
print(rolling_sharpe)

# downside_risk_

## downside_risk_1d_nb
计算 $年化下行风险 = \sqrt {\mathop E\limits_i \left[ {{{\min }^2}\left( {return{s_i} - required\_return,0} \right)} \right]}  \cdot \sqrt {ann\_factor} $

参数
- `returns` (tp.Array1d): 一维收益率时间序列
- `ann_factor` (float): 年化因子
- `required_return` (float): 目标收益率阈值，默认为0
  - 0：衡量相对于零收益的下行风险
  - 正值：衡量相对于特定收益目标的下行风险
  - 可设为无风险利率或基准收益率

返回：float，年化下行风险（正值，数值越小表示下行风险越低）

```python
@njit(cache=True)
def downside_risk_1d_nb(returns: tp.Array1d, ann_factor: float, required_return: float = 0.) -> float:
    adj_returns = returns - required_return
    adj_returns[adj_returns > 0] = 0
    return np.sqrt(np.nanmean(adj_returns ** 2)) * np.sqrt(ann_factor)
```

In [ ]:
returns = np.array([0.02, -0.01, 0.015, -0.02, 0.01])
print(downside_risk_1d_nb(returns, 252.0, 0.005))  # 日频，目标收益率0.5%

## downside_risk_nb
二维版本的 `downside_risk_1d_nb`

```python
@njit(cache=True)
def downside_risk_nb(returns: tp.Array2d, ann_factor: float, required_return: float = 0.) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = downside_risk_1d_nb(returns[:, col], ann_factor, required_return)
    return out
```

In [ ]:
returns = np.array([[0.01, 0.02, -0.005],   # 3个策略的日收益率
                     [0.015, -0.01, 0.01],
                     [-0.005, 0.03, -0.02],
                     [0.02, -0.005, 0.015]])
print(downside_risk_nb(returns, 252.0, 0.002) )

## rolling_downside_risk_nb
计算 `returns` 每列滑动窗口长度为 `window` 的年化下行风险

参数
- `returns` (tp.Array2d): 二维收益率矩阵
- `window` (int): 滚动窗口大小
- `minp` (tp.Optional[int]): 最小有效观测期数
- `ann_factor` (float): 年化因子
- `risk_free` (float): 无风险利率，默认为0
- `required_return` (float): 目标收益率，默认为0

返回值：tp.Array2d，滚动下行风险矩阵，形状与输入相同

```python
@njit
def rolling_downside_risk_nb(returns: tp.Array2d,
                             window: int,
                             minp: tp.Optional[int],
                             ann_factor: float,
                             required_return: float = 0.) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _ann_factor, _required_return):
        return downside_risk_1d_nb(_returns, _ann_factor, _required_return)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, ann_factor, required_return)
```

In [ ]:
returns = np.random.normal(0.0005, 0.015, (252, 2))  # 252交易日，2个策略
rolling_downside = rolling_downside_risk_nb(
    returns, window=30, minp=15, ann_factor=252.0, required_return=0.001)
print(rolling_downside)

# sortino_ratio_

## sortino_ratio_1d_nb
计算 $Sortino Ratio = \frac{年化超额收益}{年化下行风险}$。其中
- $年化超额收益 = \mathop E\limits_i \left[ {return{s_i} - required\_return} \right] \cdot ann\_factor$
- $年化下行风险 = \sqrt {\mathop E\limits_i \left[ {{{\min }^2}\left( {return{s_i} - required\_return,0} \right)} \right]}  \cdot \sqrt {ann\_factor} $

```python
@njit(cache=True)
def sortino_ratio_1d_nb(returns: tp.Array1d, ann_factor: float, required_return: float = 0.) -> float:
    if returns.shape[0] < 2:
        return np.nan

    adj_returns = returns - required_return
    average_annualized_return = np.nanmean(adj_returns) * ann_factor
    downside_risk = downside_risk_1d_nb(returns, ann_factor, required_return)
    if downside_risk == 0.:
        return np.inf
    return average_annualized_return / downside_risk
```

In [ ]:
returns = np.array([0.02, -0.01, 0.015, -0.005, 0.03])  # 月收益率
print(sortino_ratio_1d_nb(returns, 12.0, 0.005))

## sortino_ratio_nb
二维版本的 `sortino_ratio_1d_nb`

```python
@njit(cache=True)
def sortino_ratio_nb(returns: tp.Array2d, ann_factor: float, required_return: float = 0.) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = sortino_ratio_1d_nb(returns[:, col], ann_factor, required_return)
    return out
```

In [ ]:
returns = np.array([[0.015, 0.02, 0.01],   # 3个策略的月收益率
                     [0.01, -0.015, 0.005],
                     [-0.005, 0.025, -0.01],
                     [0.025, -0.01, 0.02]])
print(sortino_ratio_nb(returns, 12.0, 0.01))

## rolling_sortino_ratio_nb
计算 `returns` 每列滑动窗口长度为 `window` 的 Sortino ratio

参数
- `returns` (tp.Array2d): 二维收益率矩阵
- `window` (int): 滚动窗口大小
- `minp` (tp.Optional[int]): 最小有效观测期数
- `ann_factor` (float): 年化因子
- `risk_free` (float): 无风险利率，默认为0
- `required_return` (float): 目标收益率，默认为0

返回值：tp.Array2d，滚动 Sortino ratio 矩阵，形状与输入相同

```python
@njit
def rolling_sortino_ratio_nb(returns: tp.Array2d,
                             window: int,
                             minp: tp.Optional[int],
                             ann_factor: float,
                             required_return: float = 0.) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _ann_factor, _required_return):
        return sortino_ratio_1d_nb(_returns, _ann_factor, _required_return)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, ann_factor, required_return)
```

# information_ratio_

## information_ratio_1d_nb
计算 $Information Ratio = \frac{{E\left[ {returns - benchmark\_rets} \right]}}{{STD\left[ {returns - benchmark\_rets} \right]}}$

参数
- `returns` (tp.Array1d): 一维投资组合收益率时间序列
- `benchmark_rets` (tp.Array1d): 一维基准收益率时间序列
- `ddof` (int): 自由度调整，默认为1（样本标准差）

返回：float，信息比率
- 正值：策略表现优于基准，数值越大表示超额收益越稳定
- 负值：策略表现劣于基准，绝对值越大表示劣势越稳定
- 无穷大：有稳定超额收益但无主动风险（理论情况）

```python
@njit(cache=True)
def information_ratio_1d_nb(returns: tp.Array1d, benchmark_rets: tp.Array1d, ddof: int = 1) -> float:
    if returns.shape[0] < 2:
        return np.nan

    active_return = returns - benchmark_rets
    mean = np.nanmean(active_return)
    std = generic_nb.nanstd_1d_nb(active_return, ddof)
    if std == 0.:
        return np.inf
    return mean / std
```

In [ ]:
returns = np.array([0.02, 0.01, 0.015, -0.005, 0.025])  # 投资组合月收益率
benchmark = np.array([0.015, 0.008, 0.012, 0.001, 0.018])  # 基准月收益率
print(information_ratio_1d_nb(returns, benchmark, 1))

## information_ratio_nb
二维版本的 `information_ratio_1d_nb`

```python
@njit(cache=True)
def information_ratio_nb(returns: tp.Array2d, benchmark_rets: tp.Array2d, ddof: int = 1) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = information_ratio_1d_nb(returns[:, col], benchmark_rets[:, col], ddof)
    return out
```

## rolling_information_ratio_nb
计算 `returns` 每列滑动窗口长度为 `window` 的 Information Ratio

参数
- `returns` (tp.Array2d): 二维收益率矩阵
- `window` (int): 滚动窗口大小
- `minp` (tp.Optional[int]): 最小有效观测期数
- `benchmark_rets` (tp.Array2d): 二维基准收益率矩阵
- `ddof` (int): 自由度调整，默认为1

返回值：tp.Array2d，滚动 Information Ratio 矩阵，形状与输入相同

```python
@njit
def rolling_information_ratio_nb(returns: tp.Array2d,
                                 window: int,
                                 minp: tp.Optional[int],
                                 benchmark_rets: tp.Array2d,
                                 ddof: int = 1) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _benchmark_rets, _ddof):
        return information_ratio_1d_nb(_returns, _benchmark_rets[i + 1 - len(_returns):i + 1, col], _ddof)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, benchmark_rets, ddof)
```

# CAPM 模型：beta 和 alpha
$$E\left[ {{R_i}} \right] = {R_f} + {\beta _i}\left( {E\left[ {{R_m}} \right] - {R_f}} \right) + {\alpha _i}$$
- $E\left[ {{R_i}} \right]$：资产 $i$ 的期望收益率
- $R_f$：无风险收益。通常用政府债券收益率代表，确定性收益，不存在违约风险
- ${E\left[ {{R_m}} \right]}$：包含所有可投资资产的市场组合的期望收益率。例如中国的沪深300指数、中证500指数
- ${\beta _i} = \frac{{Cov\left[ {{R_i},{R_m}} \right]}}{{Var\left[ {{R_m}} \right]}}$：衡量资产 $i$ 相对于市场组合的系统性风险
- $\alpha _i$：衡量资产 $i$ 相对于市场组合的超额收益

## beta_

### beta_1d_nb
计算 $Beta = \frac{{Cov\left[ {returns,benchmark\_rets} \right]}}{{Var\left[ {benchmark\_rets} \right]}}$
- 注意函数中实现利用了 $E\left[ {\left( {X - E\left[ X \right]} \right)\left( {Y - E\left[ Y \right]} \right)} \right] = E\left[ {X\left( {Y - E\left[ Y \right]} \right)} \right] - E\left[ X \right]E\left[ {\left( {Y - E\left[ Y \right]} \right)} \right] = E\left[ {X\left( {Y - E\left[ Y \right]} \right)} \right]$

参数
- `returns` (tp.Array1d): 一维投资组合收益率时间序列
- `benchmark_rets` (tp.Array1d): 一维基准收益率时间序列

返回值：float，Beta系数
- Beta = 1：与基准波动完全一致，承担平均市场风险
- Beta > 1：比基准更加波动，属于高风险高收益类型
- Beta < 1：比基准波动更小，属于防御性投资
- Beta = 0：与基准无相关性，收益率独立于市场
- Beta < 0：与基准呈负相关，具有对冲特性

```python
@njit(cache=True)
def beta_1d_nb(returns: tp.Array1d, benchmark_rets: tp.Array1d) -> float:
    if benchmark_rets.shape[0] < 2:
        return np.nan

    independent = np.where(
        np.isnan(returns),
        np.nan,
        benchmark_rets,
    )
    ind_residual = independent - np.nanmean(independent)
    covariances = np.nanmean(ind_residual * returns)
    ind_residual = ind_residual ** 2
    ind_variances = np.nanmean(ind_residual)
    if ind_variances < 1.0e-30:
        ind_variances = np.nan
    if ind_variances == 0.:
        return np.inf
    return covariances / ind_variances
```

In [ ]:
returns = np.array([0.02, -0.01, 0.015, -0.005, 0.025])  # 投资组合月收益率
benchmark = np.array([0.018, -0.008, 0.012, -0.003, 0.02])  # 市场基准月收益率
print(beta_1d_nb(returns, benchmark))

### beta_nb
二维版本的 `beta_1d_nb`

```python
@njit(cache=True)
def beta_nb(returns: tp.Array2d, benchmark_rets: tp.Array2d) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = beta_1d_nb(returns[:, col], benchmark_rets[:, col])
    return out
```

In [ ]:
returns = np.array([[0.02, 0.015, 0.025],      # 3只股票的月收益率
                     [0.01, 0.008, -0.012],
                     [0.015, -0.005, 0.022],
                     [-0.005, 0.012, -0.008]])
benchmarks = np.array([[0.018, 0.018, 0.018],  # 市场基准收益率
                        [0.008, 0.008, 0.008],
                        [0.012, 0.012, 0.012],
                        [0.002, 0.002, 0.002]])
print(beta_nb(returns, benchmarks))

### rolling_beta_nb
计算 `returns` 每列滑动窗口长度为 `window` 的 Beta

参数
- `returns` (tp.Array2d): 二维投资组合收益率矩阵，形状为(时间点数, 资产数)
- `benchmark_rets` (tp.Array2d): 二维基准收益率矩阵，形状与returns相同

返回值：tp.Array1d，各资产Beta系数数组，形状为(资产数,)

```python
@njit
def rolling_beta_nb(returns: tp.Array2d,
                    window: int,
                    minp: tp.Optional[int],
                    benchmark_rets: tp.Array2d) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _benchmark_rets):
        return beta_1d_nb(_returns, _benchmark_rets[i + 1 - len(_returns):i + 1, col])

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, benchmark_rets)
```

## alpha_

### alpha_1d_nb
计算 $$Alpha = returns - risk\_free - \frac{{Cov\left[ {returns,benchmark\_rets} \right]}}{{Var\left[ {benchmark\_rets} \right]}}\left( {benchmark\_rets - risk\_free} \right)$$

$$年化Alpha = {\left( {E\left[ {Alpha} \right] + 1} \right)^{ann\_factor}} - 1$$

参数
- `returns` (tp.Array1d): 一维投资组合收益率时间序列
- `benchmark_rets` (tp.Array1d): 一维基准收益率时间序列（通常为市场指数）
- `ann_factor` (float): 年化因子
  - 日频数据：252（年均交易日数）
  - 月频数据：12（年均月数）
  - 周频数据：52（年均周数）
- `risk_free` (float): 无风险利率，默认为0
  - 应与收益率数据频率保持一致
  - 通常使用国债收益率或央行基准利率

返回值：float，年化阿尔法系数
- 正值：投资组合表现优于预期，创造超额收益
- 负值：投资组合表现劣于预期，未能达到基准调整后的收益
- 0：投资组合表现符合CAPM模型预期

```python
@njit(cache=True)
def alpha_1d_nb(returns: tp.Array1d,
                benchmark_rets: tp.Array1d,
                ann_factor: float,
                risk_free: float = 0.) -> float:
    if returns.shape[0] < 2:
        return np.nan

    adj_returns = returns - risk_free
    adj_benchmark_rets = benchmark_rets - risk_free
    beta = beta_1d_nb(returns, benchmark_rets)
    alpha_series = adj_returns - (beta * adj_benchmark_rets)
    return (np.nanmean(alpha_series) + 1) ** ann_factor - 1
```

In [ ]:
returns = np.array([0.02, 0.015, -0.01, 0.025, 0.008])  # 投资组合月收益率
benchmark = np.array([0.018, 0.012, -0.008, 0.02, 0.006])  # 市场基准月收益率
print(alpha_1d_nb(returns, benchmark, 12.0, 0.002))

### alpha_nb
二维版本的 `alpha_1d_nb`

```python
@njit(cache=True)
def alpha_nb(returns: tp.Array2d,
             benchmark_rets: tp.Array2d,
             ann_factor: float,
             risk_free: float = 0.) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = alpha_1d_nb(returns[:, col], benchmark_rets[:, col], ann_factor, risk_free)
    return out
```

In [ ]:
# 三个投资策略的月收益率
returns = np.array([[0.02, 0.015, -0.005],   # 策略A, B, C的第1个月收益
                     [0.01, -0.008, 0.012],   # 策略A, B, C的第2个月收益
                     [-0.005, 0.02, 0.008],   # 策略A, B, C的第3个月收益
                     [0.025, 0.01, -0.01]])   # 策略A, B, C的第4个月收益
 
# 三个策略对应的基准收益率（可以是不同基准）
benchmarks = np.array([[0.018, 0.012, -0.002],  # 基准收益率
                        [0.008, -0.005, 0.01],
                        [-0.002, 0.018, 0.006],
                        [0.02, 0.008, -0.008]])
 
print(alpha_nb(returns, benchmarks, 12.0, 0.002))  # 月频数据，无风险利率0.2%

### rolling_alpha_nb
计算 `returns` 每列滑动窗口长度为 `window` 的 Alpha

参数
- `returns` (tp.Array2d): 二维投资组合收益率矩阵，形状为(时间点数, 资产数)
- `window` (int): 滚动窗口大小（时间点数）
    - 日频数据：建议63天（一个季度）到252天（一年）
    - 月频数据：建议12个月到36个月
    - 窗口过小：统计不稳定；窗口过大：响应迟缓
- `minp` (tp.Optional[int]): 计算所需的最少观测值
    - None：使用window作为最小观测值要求
    - 整数：指定最小有效观测值数量
- `benchmark_rets` (tp.Array2d): 二维基准收益率矩阵，形状与returns相同
- `ann_factor` (float): 年化因子
- `risk_free` (float): 无风险利率，默认为0

返回值：tp.Array2d，滚动Alpha矩阵，形状与输入returns相同

```python
@njit
def rolling_alpha_nb(returns: tp.Array2d,
                     window: int,
                     minp: tp.Optional[int],
                     benchmark_rets: tp.Array2d,
                     ann_factor: float,
                     risk_free: float = 0.) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _benchmark_rets, _ann_factor, _risk_free):
        return alpha_1d_nb(_returns, _benchmark_rets[i + 1 - len(_returns):i + 1, col], _ann_factor, _risk_free)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, benchmark_rets, ann_factor, risk_free)
```

# tail_ratio_

## tail_ratio_1d_nb
计算 $Tail Ratio = \frac{{\left| {95\%分位数 } \right|}}{{\left| {5\%分位数 } \right|}}$
- `d%` 分位数：将 `returns` 从小到大排列，有 `d%` 的数据小于等于这个值，`100-d%` 的数据大于这个值（中间可以插值）

参数
- `returns` (tp.Array1d): 一维收益率时间序列

返回值：float，尾部比率（正数或无穷大）
- 比率 > 1：极端正收益超过极端损失，分布特征较优
- 比率 = 1：极端收益和损失相当，分布相对对称
- 比率 < 1：极端损失超过极端收益，分布风险偏向
- 比率 = ∞：极端损失接近零，策略几乎无下行风险

```python
@njit(cache=True)
def tail_ratio_1d_nb(returns: tp.Array1d) -> float:
    returns = returns[~np.isnan(returns)]
    if len(returns) < 1:
        return np.nan
    perc_95 = np.abs(np.percentile(returns, 95))
    perc_5 = np.abs(np.percentile(returns, 5))
    if perc_5 == 0.:
        return np.inf
    return perc_95 / perc_5
```

In [ ]:
returns = np.array([0.05, -0.03, 0.02, -0.01, 0.08, -0.02, 0.01, -0.04])
print(tail_ratio_1d_nb(returns))

## tail_ratio_nb
二维版本的 `tail_ratio_1d_nb`

```python
@njit(cache=True)
def tail_ratio_nb(returns: tp.Array2d) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = tail_ratio_1d_nb(returns[:, col])
    return out
```

## rolling_tail_ratio_nb
计算 `returns` 每列滑动窗口长度为 `window` 的 Tail Ratio

参数
- `returns` (tp.Array2d): 二维投资组合收益率矩阵，形状为(时间点数, 资产数)
- `window` (int): 滚动窗口大小（时间点数）
- `minp` (tp.Optional[int]): 计算所需的最少观测值
    - None：使用window作为最小观测值要求
    - 整数：指定最小有效观测值数量

返回值：tp.Array2d，滚动Tail Ratio矩阵，形状与输入returns相同

```python
@njit
def rolling_tail_ratio_nb(returns: tp.Array2d, window: int, minp: tp.Optional[int]) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns):
        return tail_ratio_1d_nb(_returns)
    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb)
```

# value_at_risk_

## value_at_risk_1d_nb
计算 $VaR = 100cutoff\% 分位数$
- `d%` 分位数：将 `returns` 从小到大排列，有 `d%` 的数据小于等于这个值，`100-d%` 的数据大于这个值（中间可以插值）

参数
- `returns` (tp.Array1d): 一维收益率时间序列
- `cutoff` (float)

返回值：float，VaR值

```python
@njit(cache=True)
def value_at_risk_1d_nb(returns: tp.Array1d, cutoff: float = 0.05) -> float:
    returns = returns[~np.isnan(returns)]
    if len(returns) < 1:
        return np.nan
    return np.percentile(returns, 100 * cutoff)
```

In [ ]:
# 模拟100天的日收益率数据
returns = np.array([-0.08, -0.03, 0.02, -0.01, 0.04, -0.02, 0.01, -0.05])

# 计算95% VaR (5%分位数)
var_5 = value_at_risk_1d_nb(returns, 0.05)
print(f"95% VaR: {var_5:.3f}")  # -0.080 (-8.0%)
 
# 计算99% VaR (1%分位数) 
var_1 = value_at_risk_1d_nb(returns, 0.01)  
print(f"99% VaR: {var_1:.3f}")  # -0.080 (-8.0%)

## value_at_risk_nb
二维版本的 `value_at_risk_1d_nb`

```python
@njit(cache=True)
def value_at_risk_nb(returns: tp.Array2d, cutoff: float = 0.05) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = value_at_risk_1d_nb(returns[:, col], cutoff)
    return out
```

## rolling_value_at_risk_nb
计算 `returns` 每列滑动窗口长度为 `window` 的 VaR

参数
- `returns` (tp.Array2d): 二维投资组合收益率矩阵，形状为(时间点数, 资产数)
- `window` (int): 滚动窗口大小（时间点数）
- `minp` (tp.Optional[int]): 计算所需的最少观测值
    - None：使用window作为最小观测值要求
    - 整数：指定最小有效观测值数量
- `cutoff` (float)

返回值：tp.Array2d，滚动 VaR 矩阵，形状与输入returns相同

```python
@njit
def rolling_value_at_risk_nb(returns: tp.Array2d,
                             window: int,
                             minp: tp.Optional[int],
                             cutoff: float = 0.05) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _cutoff):
        return value_at_risk_1d_nb(_returns, _cutoff)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, cutoff)
```

# cond_value_at_risk_

## cond_value_at_risk_1d_nb
计算 $CVaR = E\left[ {returns|returns < cutoff} \right]$

参数
- `returns` (tp.Array1d): 一维收益率时间序列
- `cutoff` (float)

返回值：float，CVaR值（通常为负数，表示条件期望损失）

```python
@njit(cache=True)
def cond_value_at_risk_1d_nb(returns: tp.Array1d, cutoff: float = 0.05) -> float:
    cutoff_index = int((len(returns) - 1) * cutoff)
    return np.mean(np.partition(returns, cutoff_index)[:cutoff_index + 1])
```

In [ ]:
# 模拟包含极端损失的收益率序列
returns = np.array([-0.15, -0.08, -0.05, -0.03, -0.02, 
                     0.01, 0.02, 0.03, 0.04, 0.05])
 
# 计算95% CVaR (最差5%的平均损失)
cvar_5 = cond_value_at_risk_1d_nb(returns, 0.05)
print(f"95% CVaR: {cvar_5:.3f} ({cvar_5*100:.1f}%)")

## cond_value_at_risk_nb
二维版本的 `cond_value_at_risk_1d_nb`

```python
@njit(cache=True)
def cond_value_at_risk_nb(returns: tp.Array2d, cutoff: float = 0.05) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = cond_value_at_risk_1d_nb(returns[:, col], cutoff)
    return out
```

## rolling_cond_value_at_risk_nb
计算 `returns` 每列滑动窗口长度为 `window` 的 CVaR

参数
- `returns` (tp.Array2d): 二维投资组合收益率矩阵，形状为(时间点数, 资产数)
- `window` (int): 滚动窗口大小（时间点数）
- `minp` (tp.Optional[int]): 计算所需的最少观测值
    - None：使用window作为最小观测值要求
    - 整数：指定最小有效观测值数量
- `cutoff` (float)

返回值：tp.Array2d，滚动 CVaR 矩阵，形状与输入returns相同

```python
@njit
def rolling_cond_value_at_risk_nb(returns: tp.Array2d,
                                  window: int,
                                  minp: tp.Optional[int],
                                  cutoff: float = 0.05) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _cutoff):
        return cond_value_at_risk_1d_nb(_returns, _cutoff)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, cutoff)
```

# capture_

## capture_1d_nb
计算 $Capture Ratio = \frac{returns年化收益率}{benchmark\_rets年化收益率}$
- 年化收益率计算参考 `annualized_return_`

参数
- `returns` (tp.Array1d): 一维投资组合收益率时间序列
- `benchmark_rets` (tp.Array1d): 一维基准收益率时间序列
- `ann_factor` (float): 年化因子
  - 日频数据：252（年均交易日数）
  - 月频数据：12（年均月数）
  - 周频数据：52（年均周数）

返回值：float，Capture Ratio
- \> 1.0：投资组合年化收益超过基准，表现优秀
- = 1.0：投资组合年化收益等于基准，表现持平
- < 1.0：投资组合年化收益低于基准，表现不佳
- ∞：基准收益为零而投资组合有正收益（理想情况）

```python
@njit(cache=True)
def capture_1d_nb(returns: tp.Array1d, benchmark_rets: tp.Array1d, ann_factor: float) -> float:
    annualized_return1 = annualized_return_1d_nb(returns, ann_factor)
    annualized_return2 = annualized_return_1d_nb(benchmark_rets, ann_factor)
    if annualized_return2 == 0.:
        return np.inf
    return annualized_return1 / annualized_return2
```

In [17]:
# 模拟投资组合和基准的月收益率数据（12个月）
returns = np.array([0.02, 0.015, -0.01, 0.025, 0.008, 0.012,
                     -0.005, 0.018, 0.022, -0.008, 0.015, 0.01])
benchmark = np.array([0.018, 0.012, -0.008, 0.02, 0.006, 0.01,
                      -0.003, 0.015, 0.018, -0.006, 0.012, 0.008])
 
capture_ratio = capture_1d_nb(returns, benchmark, 12.0)
print(f"捕获比率: {capture_ratio:.2f}")

捕获比率: 1.21


## capture_nb
二维版本的 `capture_1d_nb`

```python
@njit(cache=True)
def capture_nb(returns: tp.Array2d, benchmark_rets: tp.Array2d, ann_factor: float) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = capture_1d_nb(returns[:, col], benchmark_rets[:, col], ann_factor)
    return out
```

## rolling_capture_nb
计算 `returns` 每列滑动窗口长度为 `window` 的 Capture Ratio

```python
@njit
def rolling_capture_nb(returns: tp.Array2d,
                       window: int,
                       minp: tp.Optional[int],
                       benchmark_rets: tp.Array2d,
                       ann_factor: float) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _benchmark_rets, _ann_factor):
        return capture_1d_nb(_returns, _benchmark_rets[i + 1 - len(_returns):i + 1, col], _ann_factor)
    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, benchmark_rets, ann_factor)
```

# up_capture_

## up_capture_1d_nb
计算 $Up Capture Ratio = \frac{returns(benchmark\_rets > 0)年化收益率}{benchmark\_rets(benchmark\_rets > 0)年化收益率}$
- 年化收益率计算参考 `annualized_return_`

参数
- `returns` (tp.Array1d): 一维投资组合收益率时间序列
- `benchmark_rets` (tp.Array1d): 一维基准收益率时间序列
- `ann_factor` (float): 年化因子
  - 日频数据：252（年均交易日数）
  - 月频数据：12（年均月数）
  - 周频数据：52（年均周数）

返回值：float，Up Capture Ratio
- \> 1.0：在基准上涨时，投资组合涨幅更大，上行参与度高
- = 1.0：在基准上涨时，投资组合涨幅与基准一致
- < 1.0：在基准上涨时，投资组合涨幅较小，上行参与度低

```python
@njit(cache=True)
def up_capture_1d_nb(returns: tp.Array1d, benchmark_rets: tp.Array1d, ann_factor: float) -> float:
    returns = returns[benchmark_rets > 0]
    benchmark_rets = benchmark_rets[benchmark_rets > 0]
    if returns.shape[0] < 1:
        return np.nan
    annualized_return1 = annualized_return_1d_nb(returns, ann_factor)
    annualized_return2 = annualized_return_1d_nb(benchmark_rets, ann_factor)
    if annualized_return2 == 0.:
        return np.inf
    return annualized_return1 / annualized_return2
```

In [18]:
# 模拟牛熊交替的市场环境
benchmark = np.array([0.03, -0.02, 0.025, -0.01, 0.04, -0.015, 0.02])
returns = np.array([0.035, -0.015, 0.02, -0.012, 0.045, -0.018, 0.025])
 
up_capture = up_capture_1d_nb(returns, benchmark, 12.0)
print(f"上行捕获比率: {up_capture:.2f}")

上行捕获比率: 1.10


## up_capture_nb
二维版本的 `up_capture_1d_nb`

```python
@njit(cache=True)
def up_capture_nb(returns: tp.Array2d, benchmark_rets: tp.Array2d, ann_factor: float) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = up_capture_1d_nb(returns[:, col], benchmark_rets[:, col], ann_factor)
    return out
```

## rolling_up_capture_nb
计算 `returns` 每列滑动窗口长度为 `window` 的 Up Capture Ratio

```python
@njit
def rolling_up_capture_nb(returns: tp.Array2d,
                          window: int,
                          minp: tp.Optional[int],
                          benchmark_rets: tp.Array2d,
                          ann_factor: float) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _benchmark_rets, _ann_factor):
        return up_capture_1d_nb(_returns, _benchmark_rets[i + 1 - len(_returns):i + 1, col], _ann_factor)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, benchmark_rets, ann_factor)
```

# down_capture_

## down_capture_1d_nb
计算 $Down Capture Ratio = \frac{returns(benchmark\_rets < 0)年化收益率}{benchmark\_rets(benchmark\_rets < 0)年化收益率}$
- 年化收益率计算参考 `annualized_return_`

参数
- `returns` (tp.Array1d): 一维投资组合收益率时间序列
- `benchmark_rets` (tp.Array1d): 一维基准收益率时间序列
- `ann_factor` (float): 年化因子
  - 日频数据：252（年均交易日数）
  - 月频数据：12（年均月数）
  - 周频数据：52（年均周数）

返回值：float，Down Capture Ratio
- < 1.0：在基准下跌时，投资组合跌幅较小，下行保护能力强（理想）
- = 1.0：在基准下跌时，投资组合跌幅与基准一致
- \> 1.0：在基准下跌时，投资组合跌幅更大，下行保护能力弱

```python
@njit(cache=True)
def down_capture_1d_nb(returns: tp.Array1d, benchmark_rets: tp.Array1d, ann_factor: float) -> float:
    returns = returns[benchmark_rets < 0]
    benchmark_rets = benchmark_rets[benchmark_rets < 0]
    if returns.shape[0] < 1:
        return np.nan
    annualized_return1 = annualized_return_1d_nb(returns, ann_factor)
    annualized_return2 = annualized_return_1d_nb(benchmark_rets, ann_factor)
    if annualized_return2 == 0.:
        return np.inf
    return annualized_return1 / annualized_return2
```

In [19]:
# 模拟牛熊交替的市场环境
benchmark = np.array([0.03, -0.04, 0.025, -0.03, 0.02, -0.05, 0.01])
returns = np.array([0.035, -0.03, 0.02, -0.025, 0.025, -0.04, 0.015])

down_capture = down_capture_1d_nb(returns, benchmark, 12.0)
print(f"下行捕获比率: {down_capture:.2f}")

下行捕获比率: 0.83


## down_capture_nb
二维版本的 `down_capture_1d_nb`

```python
@njit(cache=True)
def down_capture_nb(returns: tp.Array2d, benchmark_rets: tp.Array2d, ann_factor: float) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = down_capture_1d_nb(returns[:, col], benchmark_rets[:, col], ann_factor)
    return out
```

## rolling_down_capture_nb
计算 `returns` 每列滑动窗口长度为 `window` 的 Down Capture Ratio

```python
@njit
def rolling_down_capture_nb(returns: tp.Array2d,
                            window: int,
                            minp: tp.Optional[int],
                            benchmark_rets: tp.Array2d,
                            ann_factor: float) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _benchmark_rets, _ann_factor):
        return down_capture_1d_nb(_returns, _benchmark_rets[i + 1 - len(_returns):i + 1, col], _ann_factor)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, benchmark_rets, ann_factor)
```